# Neural Network with Tuned Single Hidden Layer

This notebook trains a neural network with a **single hidden layer** where the number of neurons is
**optimally selected** for each rolling window using a temporal validation split.

**Approach:**
1. For each prediction month, take the rolling window (252 trading days)
2. Split temporally: first 80% of dates for training, last 20% for validation
3. Train models with each candidate neuron count, evaluate on validation set
4. Select the neuron count with lowest validation MSE
5. Retrain on the full rolling window with optimal neurons
6. Predict all stocks for the next month

In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from pathlib import Path
import time
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

In [ ]:
# Number of CPU cores for parallel processing
N_JOBS = 16

# Features and target
TARGET = 'f_cumret1'
FEATURES = ['net_sentiment', 'log_volume']

# Rolling window
WINDOW = 252  # Trading days (1 year)
TRAIN_END_DATE = '2011-12-31'

# Temporal validation split
VALIDATION_FRACTION = 0.20  # Last 20% of rolling window dates for validation

# Neuron candidates for single hidden layer
NEURON_CANDIDATES = [2, 4, 8, 16, 32, 64, 128, 256]

# Base neural network parameters
nn_base_params = {
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 'auto',
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.001,
    'max_iter': 200,
    'random_state': 42,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 10,
    'verbose': False
}

print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")
print(f"Rolling window: {WINDOW} trading days")
print(f"Validation fraction: {VALIDATION_FRACTION}")
print(f"Neuron candidates: {NEURON_CANDIDATES}")

In [ ]:
data = pd.read_pickle(INPUT_DATA)

In [ ]:
# Prepare features and target
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# OOS Predictions with Optimal Neuron Selection

In [ ]:
# Get unique dates and define OOS period
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training end: {TRAIN_END_DATE}")
print(f"OOS period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Parallel processing: {N_JOBS} cores")

In [ ]:
def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates,
                            nn_base_params, FEATURES, TARGET, WINDOW, NEURON_CANDIDATES,
                            VALIDATION_FRACTION):
    """
    For a single prediction month:
    1. Get rolling window training data (WINDOW trading days)
    2. Temporal split: first (1 - VALIDATION_FRACTION) dates for training,
       last VALIDATION_FRACTION dates for validation
    3. Select optimal neuron count by minimizing validation MSE
    4. Retrain on full rolling window with optimal neurons
    5. Predict all stocks for the prediction month

    Returns:
        predictions: list of dicts with date, index, prediction
        month_info: dict with tuning diagnostics
    """
    predictions = []
    month_info = {}

    # Get prediction dates for this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return predictions, month_info

    # Training cutoff: end of previous month
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)

    train_dates_available = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates_available) == 0:
        return predictions, month_info
    last_train_date = train_dates_available[-1]

    # Rolling window boundaries
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return predictions, month_info

    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    window_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    window_data = model_data.loc[window_mask]

    if len(window_data) == 0:
        return predictions, month_info

    # --- Step 1: Temporal split for neuron selection ---
    window_unique_dates = sorted(window_data['date'].unique())
    n_dates = len(window_unique_dates)
    split_idx = int(n_dates * (1 - VALIDATION_FRACTION))
    split_date = window_unique_dates[split_idx]

    train_sub_mask = window_data['date'] < split_date
    val_mask = window_data['date'] >= split_date

    X_train_sub = window_data.loc[train_sub_mask, FEATURES]
    y_train_sub = window_data.loc[train_sub_mask, TARGET]
    X_val = window_data.loc[val_mask, FEATURES]
    y_val = window_data.loc[val_mask, TARGET]

    if len(X_train_sub) == 0 or len(X_val) == 0:
        return predictions, month_info

    # Standardize features for tuning phase
    scaler_tune = StandardScaler()
    X_train_sub_scaled = scaler_tune.fit_transform(X_train_sub)
    X_val_scaled = scaler_tune.transform(X_val)

    # --- Step 2: Find optimal neuron count ---
    best_neurons = NEURON_CANDIDATES[0]
    best_val_mse = np.inf
    tuning_results = {}

    for n_neurons in NEURON_CANDIDATES:
        params = nn_base_params.copy()
        params['hidden_layer_sizes'] = (n_neurons,)

        try:
            model = MLPRegressor(**params)
            model.fit(X_train_sub_scaled, y_train_sub)
            val_pred = model.predict(X_val_scaled)
            val_mse = mean_squared_error(y_val, val_pred)
        except Exception:
            val_mse = np.inf

        tuning_results[n_neurons] = val_mse

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_neurons = n_neurons

    # --- Step 3: Retrain on full rolling window with optimal neurons ---
    X_full = window_data[FEATURES]
    y_full = window_data[TARGET]

    scaler_final = StandardScaler()
    X_full_scaled = scaler_final.fit_transform(X_full)

    final_params = nn_base_params.copy()
    final_params['hidden_layer_sizes'] = (best_neurons,)

    final_model = MLPRegressor(**final_params)
    final_model.fit(X_full_scaled, y_full)

    # --- Step 4: Predict for all days in the month ---
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]

        if len(X_test) == 0:
            continue

        test_indices = model_data.index[test_mask]
        X_test_scaled = scaler_final.transform(X_test)
        y_pred = final_model.predict(X_test_scaled)

        for idx, pred in zip(test_indices, y_pred):
            predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })

    month_info = {
        'month': str(pred_month),
        'best_neurons': best_neurons,
        'best_val_mse': best_val_mse,
        'tuning_results': tuning_results,
        'train_samples': len(X_train_sub),
        'val_samples': len(X_val),
        'full_train_samples': len(X_full),
        'n_predictions': len(predictions)
    }

    return predictions, month_info

In [ ]:
# Run parallel processing across months
print(f"Starting parallel processing with {N_JOBS} jobs...")
print(f"Processing {len(oos_months)} months...")
print(f"Each month: testing {len(NEURON_CANDIDATES)} neuron candidates + 1 final retrain")

start_time = time.time()

def get_month_slice(pred_month):
    """Pre-slice model_data to the rows needed for this month's rolling window + predictions,
    so each parallel worker receives a small DataFrame instead of a full copy of model_data."""
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return model_data.iloc[0:0]
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return model_data.iloc[0:0]
    last_train_date = train_dates[-1]
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return model_data.iloc[0:0]
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    end_date = month_dates.max()
    return model_data.loc[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)]

all_results = Parallel(n_jobs=N_JOBS, verbose=10, backend='threading')(
    delayed(train_and_predict_month)(
        month_idx, pred_month, get_month_slice(pred_month), unique_dates, oos_dates,
        nn_base_params, FEATURES, TARGET, WINDOW, NEURON_CANDIDATES,
        VALIDATION_FRACTION
    )
    for month_idx, pred_month in enumerate(oos_months)
)

elapsed = time.time() - start_time

# Separate predictions and tuning info
all_predictions = []
all_month_info = []
for preds, info in all_results:
    all_predictions.extend(preds)
    if info:
        all_month_info.append(info)

print(f"\nCompleted in {elapsed / 60:.1f} minutes.")
print(f"Total predictions: {len(all_predictions):,}")
print(f"Months with results: {len(all_month_info)}")

# Neuron Selection Analysis

In [ ]:
# Build tuning results DataFrame
tuning_df = pd.DataFrame(all_month_info)

print("Optimal Neuron Selection Summary")
print("=" * 50)
print("\nFrequency of each optimal neuron count:")
print(tuning_df['best_neurons'].value_counts().sort_index())
print(f"\nMean optimal neurons: {tuning_df['best_neurons'].mean():.1f}")
print(f"Median optimal neurons: {tuning_df['best_neurons'].median():.1f}")
print(f"\nSample sizes per month (average):")
print(f"  Training (tuning):  {tuning_df['train_samples'].mean():,.0f}")
print(f"  Validation:         {tuning_df['val_samples'].mean():,.0f}")
print(f"  Full window:        {tuning_df['full_train_samples'].mean():,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Distribution of optimal neuron counts
neuron_counts = tuning_df['best_neurons'].value_counts().sort_index()
axes[0].bar(neuron_counts.index.astype(str), neuron_counts.values, color='steelblue')
axes[0].set_xlabel('Number of Neurons')
axes[0].set_ylabel('Frequency (months)')
axes[0].set_title('Distribution of Optimal Neuron Count')

# Plot 2: Optimal neuron count over time
month_dates = pd.to_datetime(tuning_df['month'].values)
axes[1].plot(month_dates, tuning_df['best_neurons'].values,
             marker='o', markersize=3, linewidth=1, color='steelblue')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Optimal Neurons')
axes[1].set_title('Optimal Neuron Count Over Time')
axes[1].set_yticks([1, 2, 4, 8, 16, 32, 64, 128])

# Plot 3: Average validation MSE vs neuron count
mse_data = pd.DataFrame(tuning_df['tuning_results'].tolist())
avg_mse = mse_data.replace(np.inf, np.nan).mean()
axes[2].plot(avg_mse.index, avg_mse.values, marker='o', color='steelblue')
axes[2].set_xlabel('Number of Neurons')
axes[2].set_ylabel('Average Validation MSE')
axes[2].set_title('Validation MSE vs Neuron Count')
axes[2].set_xscale('log', base=2)
axes[2].set_xticks([1, 2, 4, 8, 16, 32, 64, 128])
axes[2].set_xticklabels([1, 2, 4, 8, 16, 32, 64, 128])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nn_tuning_analysis_1layer.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved to Figures/nn_tuning_analysis_1layer.png")

# Save Results

In [ ]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(all_predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nPrediction statistics:")
print(predictions_df['prediction'].describe())

# Check prediction variance (the old model predicted constants)
print(f"\nPrediction variance check:")
sample_date = predictions_df['date'].iloc[0]
sample_preds = predictions_df.loc[predictions_df['date'] == sample_date, 'prediction']
print(f"  Date: {sample_date}")
print(f"  N stocks: {len(sample_preds)}")
print(f"  Unique predictions: {sample_preds.nunique()}")
print(f"  Std of predictions: {sample_preds.std():.8f}")

In [ ]:
# Save predictions
OUTPUT_FILE = MODEL_DATA_DIR / f"predictions_neural_network_tuned_1layer_input={len(FEATURES)}.pkl"
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")

# Save tuning results
TUNING_FILE = MODEL_DATA_DIR / f"nn_tuning_results_1layer_input={len(FEATURES)}.pkl"
tuning_df.to_pickle(TUNING_FILE)

print(f"\nTuning results saved to: {TUNING_FILE}")
print(f"Tuning file size: {TUNING_FILE.stat().st_size / (1024**2):.4f} MB")